# 🗡️ CTE (WITH 语法) 与复杂 SQL 连招

> **大白话**：CTE（Common Table Expressions）就是 SQL 里的“备菜盘”或者“中间变量”。
它让你能把一个无比巨大的 SQL，拆解成**逻辑清晰的 1步、2步、3步**，最后再拼起来。

### 为什么需要 CTE？
假设老板提了一个魔鬼需求：**“找出今年消费总额排名前3的 VIP 客户，并提取出他们买过的所有商品明细。”**

如果不用 CTE，你必须用**嵌套子查询**，也就是 `SELECT * FROM (SELECT * FROM (SELECT ...))`。
这就像俄罗斯套娃，如果嵌套了三四层，除了写代码的那个人，其他谁也看不懂。

而用 CTE（`WITH`），你的思路就会变成一条清晰的流水线（连招）！

In [1]:
import pandas as pd
import sqlite3

# ==========================================
# 🔧 准备环境：创建一个内存里的数据库来模拟实战
# ==========================================
conn = sqlite3.connect(':memory:')

# 模拟一份有点杂乱的订单表
data = {
    '用户ID': ['U1', 'U1', 'U2', 'U2', 'U3', 'U3', 'U3', 'U4'],
    '订单ID': ['O1', 'O2', 'O3', 'O4', 'O5', 'O6', 'O7', 'O8'],
    '商品': ['外星人电脑', '鼠标', '键盘', '显示器', '微单', '镜头', '电池', 'U盘'],
    '金额': [15000, 200, 300, 1500, 9000, 4500, 300, 50]
}
df_mock = pd.DataFrame(data)
df_mock.to_sql('orders', conn, index=False, if_exists='replace')

# 看看我们的原始数据
df_mock

,用户ID,订单ID,商品,金额
0,U1,O1,外星人电脑,15000
1,U1,O2,鼠标,200
2,U2,O3,键盘,300
3,U2,O4,显示器,1500
4,U3,O5,微单,9000
5,U3,O6,镜头,4500
6,U3,O7,电池,300
7,U4,O8,U盘,50


### 实战连招：找出**消费最高的前 2 名用户**，并拉出他们全部的**购买明细**！

**解题口诀（3个备菜盘）：**
1. 备菜盘 A `Total_Sales`：帮我把每个用户的总金额算出来。
2. 备菜盘 B `Top_VIPs`：从盘 A 里面只挑出金额最大的 2 个人。
3. 下锅炒菜 `SELECT ... `：拿原始数据表，跟盘 B (也就是 VIP 名单) 做一个内连接 (INNER JOIN)。

In [2]:
# 我们用 WITH 把这个三步连招干脆利落地写出来！
sql_query = """
-- 起手式：WITH 关键字
WITH 
-- 🔪 备菜盘 A（第一重变量）：算出每个用户的总消费
Total_Sales AS (
    SELECT 用户ID, SUM(金额) AS 消费总额
    FROM orders
    GROUP BY 用户ID
),

-- 🔪 备菜盘 B（第二重变量）：基于盘A，取出前2名
Top_VIPs AS (
    SELECT 用户ID 
    FROM Total_Sales
    ORDER BY 消费总额 DESC
    LIMIT 2
)

-- 🍳 热油下锅（核心查询）：原表与盘B匹配，只留下那2个大哥的订单明细！
SELECT 
    o.用户ID, 
    o.订单ID, 
    o.商品, 
    o.金额
FROM orders o
INNER JOIN Top_VIPs v 
    ON o.用户ID = v.用户ID;
"""

# Pandas 提供了一个直接执行 SQL 的函数，结果变回 DataFrame
result_df = pd.read_sql_query(sql_query, conn)
result_df

,用户ID,订单ID,商品,金额
0,U1,O1,外星人电脑,15000
1,U1,O2,鼠标,200
2,U3,O5,微单,9000
3,U3,O6,镜头,4500
4,U3,O7,电池,300


### 脑洞对比：如果用 Pandas 来写同样逻辑，是不是感觉一模一样？

在写 Python/Pandas 的时候，我们天生就会通过“给变量赋值”来接成连招。而 **CTE (`WITH`) 就是为了让 SQL 也能像写 Python 一样声明变量！**

In [3]:
# 刀法一（对应备菜盘 A）：求每个用户消费总额
total_sales = df_mock.groupby('用户ID')['金额'].sum().reset_index()

# 刀法二（对应备菜盘 B）：找排名前2的 VIP 榜单
top_vips = total_sales.sort_values(by='金额', ascending=False).head(2)

# 热油下锅（对应 SELECT ... INNER JOIN）：把 VIP 榜单和原表通过 用户ID 合并 (匹配) 起来
final_result = pd.merge(df_mock, top_vips[['用户ID']], on='用户ID', how='inner')

final_result

,用户ID,订单ID,商品,金额
0,U1,O1,外星人电脑,15000
1,U1,O2,鼠标,200
2,U3,O5,微单,9000
3,U3,O6,镜头,4500
4,U3,O7,电池,300
